In [29]:
from pygom import Transition, Event, TransitionType

from pygom.model.transition import InputStateError

import pytest

# Water synthesis:
trans_H = Transition(origin="H2", magnitude='2')
trans_O = Transition(origin="O2")
trans_W = Transition(destination='W', magnitude='2')

event_water_synthesis = Event(
    rate='r',
    transition_list=[trans_H, trans_O, trans_W]
)

assert len(event_water_synthesis.transition_list) == 3
assert event_water_synthesis.transition_list[0].magnitude == "2"
assert event_water_synthesis.transition_list[2].magnitude == "2"

In [30]:
# SIR model:
event_inf = Event.single_transition(origin="S", destination="I", rate="beta*S*I/N")
event_rec = Event.single_transition(origin="I", destination="R", rate="beta*S*I/N")

In [31]:
with pytest.raises(InputStateError):
    Transition()

with pytest.raises(InputStateError):
    Transition(origin="S", destination="S")

with pytest.raises(InputStateError):
    Transition(
        origin="S",
        destination="I",
        transition_type="D"
    )

t = Transition(
    origin="S",
    destination="I",
    transition_type="t"
)

assert t.transition_type == TransitionType.T

In [32]:
t1 = Transition(origin="I", destination="R")

e1 = Event(
    rate="gamma*I",
    transition_list=[t1]
)

e2 = Event.single_transition(
    rate="gamma*I",
    origin="I",
    destination="R"
)

assert e1.transition_list[0] == e2.transition_list[0]

In [33]:
# with pytest.raises(ValueError):
#     Event(
#         rate="gamma*I",
#         transition_list = [
#             Transition(origin="I", destination="R")
#         ],
#         origin="S",
#         destination="I"
#     )

In [34]:
birth = Transition(destination="S")
assert birth.transition_type == TransitionType.B

death = Transition(origin="I")
assert death.transition_type == TransitionType.D

In [35]:
# with pytest.raises(ValueError):
#     Event(
#         rate="r",
#         transition_list=[]
#     )

In [36]:
with pytest.raises(InputStateError):
    Event.single_transition(
        rate="gamma*I",
        origin="I",
        transition_type="B"
    )

In [ ]:
import numpy as np

from sympy.functions.elementary.exponential import (
    exp_polar,
    exp,
    log,
    LambertW
)
ln = log

from sympy.functions.elementary.trigonometric import (
    sin, cos, tan,
    sec, csc, cot, 
    sinc, 
    asin, acos, atan, 
    asec, acsc, acot, 
    atan2
)
arcsin = asin
arccos = acos
arctan = atan
arcsec = asec
arccsc = acsc
arccot = acot

from sympy.functions.elementary.hyperbolic import (
    sinh, cosh, tanh, 
    sech, csch, coth,
    asinh, acosh, atanh,
    acoth, asech)
arcsinh = asinh
arccosh = acosh
arctanh = atanh
arcsech = asech
# arccsch = acsch
arccoth = acoth

from sympy.functions.elementary.piecewise import Piecewise, piecewise_fold        
from sympy.functions.combinatorial.factorials import (
    factorial, factorial2, rf, ff, binomial, RisingFactorial, FallingFactorial, subfactorial
)
from sympy.functions.elementary.integers import floor, ceiling, frac
from sympy.functions.elementary.miscellaneous import (
    sqrt, root, Min, Max,
    Id, real_root, cbrt)
from sympy.functions.elementary.complexes import Abs
from sympy.core.numbers import pi
from sympy import simplify, symbols


from collections.abc import Iterable

from sympy import Expr
from sympy.parsing.sympy_parser import parse_expr

"""
TODO: 
1) Make imports of sympy functions cleaner
2) Potentially make checkEquation independent of model class object (e.g. calls to ode.params and states)
3) Currently, see how params and states are all stored.
"""

def checkEquation(
        input_str,
        state_dict,
        param_dict,
        derived_dict,
        subs_derived=True
    ) -> list:
    """
    Convert a string into an equation using the symbols from the system and 
    checks its validity. 

    Parameters
    ----------
    input_str: a str or list of str giving the equation
    ode: the parent ode
    subs_derived: should the derived parameters be substututed in?

    Returns
    -------
    A single sympy equation or list of sympy equations (depending on if 
    input_str is a single string or a list) made from the string(s)
    """

    if isinstance(input_str, str):
        input_str = [input_str]
    elif not isinstance(input_str, Iterable):
        raise TypeError("Expected a string or iterable of strings")

    namespace = param_dict | state_dict | derived_dict
    
    equations = list()
    for expr_str in input_str:
        if not isinstance(expr_str, str):
            raise TypeError("Equation should be in string format")

        try:
            eqn = parse_expr(expr_str, namespace)
        except Exception as exc:
            raise ValueError(
                f"Failed to parse equation '{expr_str}'"
            ) from exc
        
        if subs_derived and hasattr(eqn, "subs"):
            eqn = eqn.subs(derived_dict)

        equations.append(eqn)

    if len(equations) == 1:
        return equations[0]
    else:
        return equations

In [ ]:


def check_equation(
    input_str,
    state_dict,
    param_dict,
    derived_dict,
    subs_derived=True,
):
    if isinstance(input_str, str):
        input_str = [input_str]
    elif not isinstance(input_str, Iterable):
        raise TypeError("Expected a string or iterable of strings")

    namespace = param_dict | state_dict | derived_dict

    equations = []
    for expr_str in input_str:
        if not isinstance(expr_str, str):
            raise TypeError("Equation should be in string format")

        try:
            eqn = parse_expr(expr_str, namespace)
        except Exception as exc:
            raise ValueError(
                f"Failed to parse equation '{expr_str}'"
            ) from exc

        if subs_derived and hasattr(eqn, "subs"):
            eqn = eqn.subs(derived_dict)

        equations.append(eqn)

    return equations[0] if len(equations) == 1 else equations